# Advanced GraphFleet Features

This notebook demonstrates advanced features of GraphFleet:
1. Batch querying
2. Query analysis
3. Knowledge graph statistics
4. Custom pipelines

In [ ]:
from pathlib import Path
from graphfleet.core import GraphFleet
from graphfleet.core.features import GraphFleetFeatures

# Initialize GraphFleet
project_dir = Path("./data")
graph_fleet = GraphFleet(project_dir)
features = GraphFleetFeatures(graph_fleet.storage)

## Batch Querying

Process multiple queries efficiently in batches.

In [ ]:
# Prepare multiple queries
queries = [
    "What is artificial intelligence?",
    "How does machine learning work?",
    "What are neural networks?"
]

# Process queries in batch
results = await features.batch_query(
    queries,
    query_type="standard",
    batch_size=2
)

for query, result in zip(queries, results):
    print(f"Query: {query}")
    print(f"Result: {result}\n")

## Query Analysis

Analyze query results in detail.

In [ ]:
# Perform detailed query analysis
analysis = await features.query_analysis(
    "What are the main applications of AI?"
)

print("Confidence Scores:")
print(analysis["confidence_scores"])

print("\nSource Diversity:")
print(analysis["source_diversity"])

print("\nContext Analysis:")
print(analysis["context_relevance"])

## Knowledge Graph Statistics

Explore statistics about your knowledge graph.

In [ ]:
# Get knowledge graph statistics
stats = await features.knowledge_graph_stats()

print("Graph Statistics:")
print(f"Nodes: {stats['node_count']}")
print(f"Edges: {stats['edge_count']}")

print("\nCommunity Statistics:")
print(stats['community_stats'])

print("\nEmbedding Statistics:")
print(stats['embedding_stats'])

## Custom Pipelines

Create custom processing pipelines for specific use cases.

In [ ]:
# Example: Create a pipeline that combines local and global context
async def hybrid_query(query_text: str, local_weight: float = 0.7):
    # Get local and global results
    local_result = await graph_fleet.query_local(query_text)
    global_result = await graph_fleet.query_global(query_text)
    
    # Combine results based on weights
    combined_score = {}
    for doc in local_result.get("documents", []):
        combined_score[doc["id"]] = doc["score"] * local_weight
    
    for doc in global_result.get("documents", []):
        doc_id = doc["id"]
        if doc_id in combined_score:
            combined_score[doc_id] += doc["score"] * (1 - local_weight)
        else:
            combined_score[doc_id] = doc["score"] * (1 - local_weight)
    
    return sorted(
        combined_score.items(),
        key=lambda x: x[1],
        reverse=True
    )

# Try the hybrid query
results = await hybrid_query(
    "What are the practical applications of AI in healthcare?",
    local_weight=0.6
)
print("Hybrid Query Results:")
for doc_id, score in results:
    print(f"Document {doc_id}: Score {score:.3f}")